# KGC-MPC: Schur-Stable Bilinear Koopman Control & Safety-Constrained MPC (GPU Benchmark)

Self-contained Google Colab notebook to reproduce **KGC-MPC** from the public repository [`https://github.com/dashitongzhi/KGC-MPC`](https://github.com/dashitongzhi/KGC-MPC) across Norman 2019 (91,205 CRISPRa cells), Adamson 2016 (68,603 CRISPRi cells), Sci-Plex3 (762,795 cells across A549/MCF7/K562, 188 drugs × 4 doses), Sci-Plex4 (98,397 HDACi dual-drug cells), real-time FISTA-AL vs. Clarabel/SCS scaling, and N=20 closed-loop Tube-MPC stress ablations, with optional Google Drive sync.

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path('/content/KGC-MPC')
if not REPO_DIR.exists():
    !git clone https://github.com/dashitongzhi/KGC-MPC.git /content/KGC-MPC
%cd /content/KGC-MPC

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT = Path('/content/drive/MyDrive/Paper/Paper3_KGC-MPC_Koopman_Perturbation_Control')
    DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
    print('Google Drive sync target:', DRIVE_PROJECT)
except Exception as exc:
    DRIVE_PROJECT = None
    print('Drive mount skipped:', exc)

In [ ]:
!nvidia-smi -L
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
!pip -q install -r requirements.txt

In [ ]:
!python3 -B tests/smoke_test.py

In [ ]:
!python3 -u src/run_kgc_suite.py --stage all
!python3 -u src/run_tcst_benchmarks.py
!python3 -B src/make_assets.py
!python3 -B src/verify_paper.py

In [ ]:
if DRIVE_PROJECT is not None:
    for sub in ['results', 'weights']:
        src_dir = REPO_DIR / sub
        dst_dir = DRIVE_PROJECT / sub
        dst_dir.mkdir(parents=True, exist_ok=True)
        if src_dir.exists():
            for f in sorted(src_dir.glob('*')):
                if f.is_file():
                    shutil.copy2(f, dst_dir / f.name)
                    print(f'Synced {sub}/{f.name} ({f.stat().st_size:,} bytes)')
else:
    print('Drive unavailable; outputs remain under /content/KGC-MPC/.')